# Spark on the cluster from a local notebook (Livy + sparkmagic)

This notebook runs on **your laptop** with a plain Python 3 kernel. `sparkmagic` opens a **Livy session** on the cluster
and sends the code of every `%%spark` cell to it as a *statement* — the same REST calls you made with `curl` in the lab.
The Spark code runs in a YARN container, as **you**; your laptop only receives small results.

Run the cells from top to bottom. Close the session in the last cell, so it does not keep cluster resources.

## 1. Connection data

The URL comes from your handout (no trailing slash). The password is asked with `getpass`: it is kept in memory
of this kernel only and never written into the notebook file.

In [ ]:
import getpass

livy_url = input("Livy URL (from your handout, no trailing slash): ").strip().rstrip("/")
ws_user = input("User name (WORKSHOP_USER): ").strip()
ldap_password = getpass.getpass(f"LDAP password for {ws_user}: ")
workshop_root = input("WORKSHOP_ROOT [/2026_cdp_workshop]: ").strip() or "/2026_cdp_workshop"
workshop_home = f"/user/{ws_user}"

## 2. Load `sparkmagic` and open a session

`%spark add` is the `POST /sessions` of the lab. Authentication type `Basic_Access` means: LDAP user and password,
sent over HTTPS to the Knox gateway. Starting the session takes about a minute; a progress bar shows `starting` and then `idle`.

In [ ]:
%load_ext sparkmagic.magics

In [ ]:
%spark add -s workshop -l python -u $livy_url -a $ws_user -p $ldap_password -t Basic_Access -k

`%spark info` lists your session: the **Livy session id**, the **YARN application id**, and links to the Spark UI and the driver log.
Compare the application id with `yarn application -list` on the edge node.

In [ ]:
%spark info

## 3. Code on the cluster

Everything in a `%%spark` cell is Python **on the cluster**. `spark` is the `SparkSession` of the session.
Check where the cell really runs and which user it runs as.

In [ ]:
%%spark -s workshop
import getpass, socket
print("host :", socket.gethostname())
print("user :", getpass.getuser())
print("Spark:", spark.version)

## 4. Send local variables to the cluster

The variables `workshop_root` and `workshop_home` exist only in this notebook, not in the Spark driver.
`%%send_to_spark` copies them over under the same name (the cell body stays empty).

In [ ]:
%%send_to_spark -i workshop_root -t str -n workshop_root

In [ ]:
%%send_to_spark -i workshop_home -t str -n workshop_home

## 5. The pipeline from the lab, step by step

Read meter and weather CSVs with explicit schemas. Expect **15** meter rows and **10** weather rows.

In [ ]:
%%spark -s workshop
from pyspark.sql import functions as F

meter = (spark.read.option("header", True)
    .schema(
        "ts_utc STRING, station_id STRING, device_id STRING, phase STRING, "
        "active_power_kw DOUBLE, reactive_power_kvar DOUBLE, voltage_v DOUBLE, "
        "current_a DOUBLE, power_factor DOUBLE, frequency_hz DOUBLE, "
        "transformer_load_pct DOUBLE, quality_flag STRING, event_code STRING, "
        "comm_status STRING")
    .csv(workshop_root + "/data/landisgyr/sample-landisgyr-data.csv")
    .withColumn("ts_utc", F.to_timestamp(F.col("ts_utc"))))
weather = (spark.read.option("header", True)
    .schema(
        "ts_utc STRING, weather_station_id STRING, station_name STRING, "
        "temperature_c DOUBLE, relative_humidity_pct INT, wind_speed_mps DOUBLE, "
        "precipitation_mm DOUBLE, pressure_hpa DOUBLE, weather_quality STRING")
    .csv(workshop_root + "/data/operator_weather/sample-operator-weather.csv")
    .withColumn("ts_utc", F.to_timestamp(F.col("ts_utc"))))
print("meter rows:", meter.count(), "| weather rows:", weather.count())

Cleanse (keep `OK` readings) and publish the result as a temporary view, so the next cell can use plain SQL.
Expect **12** cleansed rows. The variables from the previous cell are still there — it is the same session.

In [ ]:
%%spark -s workshop
meter_ok = (meter.filter(F.col("quality_flag") == "OK")
    .dropDuplicates(["ts_utc", "station_id", "device_id", "phase"]))
meter_ok.createOrReplaceTempView("meter_ok")
print("cleansed rows:", meter_ok.count())

## 6. A SQL cell

`-c sql` turns the cell into a Spark SQL query. The result comes back as a table.

In [ ]:
%%spark -s workshop -c sql
SELECT station_id, COUNT(*) AS readings, ROUND(AVG(active_power_kw), 2) AS avg_kw, MAX(transformer_load_pct) AS max_load_pct
FROM meter_ok
GROUP BY station_id
ORDER BY station_id

## 7. Bring a small result to your laptop

`-o hourly` copies the DataFrame `hourly` from the cluster into a **pandas** DataFrame with the same name on your laptop
(at most 100 rows here). Only do this for **small** results — a large table would travel through Knox and Livy as JSON.

In [ ]:
%%spark -s workshop -o hourly -m take -n 100
hourly = (meter_ok.withColumn("hour_utc", F.date_format(F.date_trunc("hour", F.col("ts_utc")), "yyyy-MM-dd HH:mm"))
    .groupBy("station_id", "hour_utc")
    .agg(F.avg("active_power_kw").alias("avg_power_kw"), F.max("transformer_load_pct").alias("max_load_pct"))
    .orderBy("station_id", "hour_utc"))

This cell runs **locally** on pandas and matplotlib — no cluster involved. Expect two hour buckets per station.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

print(type(hourly))
hourly["label"] = hourly["station_id"] + " " + hourly["hour_utc"].str[11:16]
hourly.plot.bar(x="label", y="avg_power_kw", legend=False, figsize=(9, 4), title="Average active power per station and hour")
plt.ylabel("kW")
plt.show()

## 8. Join, write, and verify

Join weather on the timestamp (two stations per timestamp, so **24** rows), and write Parquet to your HDFS home.
Afterwards check the folder on the edge node: `hdfs dfs -ls /user/<your_user>/livy-nb-curated`.

In [ ]:
%%spark -s workshop
curated = meter_ok.join(weather, ["ts_utc"], "inner")
print("curated rows:", curated.count())
curated.write.mode("overwrite").partitionBy("station_id").parquet(workshop_home + "/livy-nb-curated")

## 9. Close the session

Closing the notebook does **not** stop the Livy session — the Spark application keeps running on YARN until Livy's idle timeout.
`%spark delete` is the `DELETE /sessions/<id>` of the lab. Run it every time.

In [ ]:
%spark delete -s workshop

Optional: `%spark cleanup` deletes **all** sessions that this kernel created, in case you opened more than one.

**Question:** Which cells were executed on your laptop, and which on the cluster? What would happen if you used
`workshop_home` in a `%%spark` cell without section 4?